This notebook calls language models through [OpenRouter](https://openrouter.ai), a service that gives you one API key for many different models. You can find model names on [openrouter.ai/models](https://openrouter.ai/models).

- `ChatOpenRouter`: the LangChain wrapper that sends our requests to OpenRouter.
- The first code cell loads your `OPENROUTER_API_KEY` from a `.env` file, or asks you for it.
- `MODEL` (optional, in your `.env` file): the model ID to use. If it is not set, the notebook uses `openai/gpt-6-luna`.

In [1]:
import os
import getpass
import warnings
from dotenv import load_dotenv

load_dotenv()
if not os.getenv("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("Enter your OpenRouter API key: ")
warnings.simplefilter(action='ignore', category=FutureWarning)

gpt_model = os.getenv("MODEL", "openai/gpt-6-luna")

In [2]:
# Using OpenRouter
from langchain_openrouter import ChatOpenRouter
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

def run_llm_chain(model_id: str,
                  prompt:PromptTemplate,
                  input: str,
                  temperature: float = 0.25) -> str:
    """
    Builds a ChatOpenRouter model and runs the chain `prompt | llm | StrOutputParser()`.

    Args:
    model_id (str): The OpenRouter model ID, e.g. "openai/gpt-6-luna".
    prompt (PromptTemplate): The prompt template, with an {input} variable.
    input (str): The text passed into the {input} variable of the prompt.
    temperature (float, optional): The temperature for text generation. Default is 0.25.

    Returns:
    str: The model's response as plain text.
    """

    llm = ChatOpenRouter(
          model=model_id,  
          temperature=temperature
      )
    llm_chain = prompt | llm | StrOutputParser()
    return llm_chain.invoke({"input":input})

/Users/moche/IADT_Local/26_27/CC-AIEng/.venv/lib/python3.14/site-packages/langchain_core/utils/pydantic.py:42: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1 import BaseModel as BaseModelV1


# 🎯 **Clarity is Essential**

- Avoid ambiguity for precise model responses.

- Use distinct delimiters (triple quotes, backticks, angle brackets) to separate instructions clearly. Cells 4 and 22 wrap the text in backticks (```` ``` ````), and cell 16 wraps it in angle brackets (`<>`).

- This prevents confusion between directives and general content.

In [3]:
text_to_summarize = f"""It's important to give clear and specific instructions \
when asking a model to perform a task. This will help guide the model towards \
the desired output, and minimize the chances of receiving irrelevant or \
incorrect responses. It's important to note that writing a clear prompt  \
doesn't necessarily mean writing a short one. In fact, longer prompts can  \
often provide more clarity and context for the model, resulting in more \
detailed and relevant outputs.
"""

template = """Summarize the text delimited by triple backticks into a single sentence: ```{input}```.
Summary:
"""

In [4]:
prompt = PromptTemplate.from_examples(
    examples=[template],
    input_variables=["input"],
    prefix = "",
    suffix=""
    )

In [5]:
prompt.pretty_print()



Summarize the text delimited by triple backticks into a single sentence: ```{input}```.
Summary:





In [6]:
result = run_llm_chain(model_id = gpt_model,
              prompt = prompt,
              input = text_to_summarize)

In [7]:
print(result)

Clear, specific prompts—whether short or long—help models produce more relevant and accurate results, and added context can improve output quality.


# 🔍 **Request Structured Output for Easy Interpretation**

- Prioritize clarity: Ask for outputs in structured formats. Cells 10 to 14 show a JSON example.

- Structured data is simpler to interpret and use. The model's reply is still plain text: this notebook prints it but does not parse it. In your own code, you would convert the text into data with a parser such as `json.loads`.

- Defining the desired format upfront leads to more organized
results.

In [8]:
text = """Create some imaginary movie titles, complete with directors and genres.
Format the information in JSON, using the keys: 'movie_id', 'title', 'director', and 'genre'.
"""


In [9]:
prompt = PromptTemplate.from_examples(
    examples=[text],
    input_variables=[],
    prefix = "",
    suffix=""
    )

In [10]:
prompt.pretty_print()



Create some imaginary movie titles, complete with directors and genres.
Format the information in JSON, using the keys: 'movie_id', 'title', 'director', and 'genre'.





In [11]:
result = run_llm_chain(model_id = gpt_model,
              prompt = prompt,
              input = text)

In [12]:
print(result)

[
  {
    "movie_id": 1,
    "title": "The Clockwork Orchard",
    "director": "Mira Solenne",
    "genre": "Fantasy"
  },
  {
    "movie_id": 2,
    "title": "Signal Beneath the Ice",
    "director": "Jonah Vale",
    "genre": "Science Fiction Thriller"
  },
  {
    "movie_id": 3,
    "title": "A Map of Small Things",
    "director": "Elise Navarro",
    "genre": "Drama"
  },
  {
    "movie_id": 4,
    "title": "Midnight at the Moon Motel",
    "director": "Theo March",
    "genre": "Mystery Comedy"
  },
  {
    "movie_id": 5,
    "title": "The Last Lantern Keeper",
    "director": "Amara Voss",
    "genre": "Adventure"
  }
]


# 🔍 **Process for Verifying Conditions**

1. **Spot the Criteria**: Look for specific requirements. In cell 16, the criterion is: does the text contain a procedural guide?

2. **Assess Compliance**: Examine the text to see if it meets these criteria. The password text in cell 16 describes steps to follow, so it meets the criterion.

3. **Customized Reply**: React accordingly. If the text is a guide, cell 16 asks for a numbered list. Otherwise it asks for the reply `No procedural guide found.` The example only triggers the first route, so the second route is not tested in this notebook.

4. **Transparent Feedback**: Ask the model to say whether the conditions were met and why. The prompt in cell 16 does not ask for this, so its output does not show this step.

In [13]:
text ="""To reset your password, first, navigate to the settings page. Click \
on the 'Security' tab, and then select 'Change Password.' Enter your current \
password, followed by your new password twice for confirmation. Click on the \
'Save Changes' button to update your password.
"""

template = """You will be provided with text enclosed in angle brackets (<>). Determine \
if the text contains a procedural guide.  If it does, convert the guide into a \
numbered list format with a title:

<Title>

1. ...
2. ...
3. ...
4. ...

If the text does not contain a procedural guide, respond with 'No procedural guide found.'

<{input}>
"""


In [14]:
prompt = PromptTemplate.from_examples(
    examples=[template],
    input_variables=["input"],
    prefix = "",
    suffix=""
    )

In [15]:
prompt.pretty_print()



You will be provided with text enclosed in angle brackets (<>). Determine if the text contains a procedural guide.  If it does, convert the guide into a numbered list format with a title:

<Title>

1. ...
2. ...
3. ...
4. ...

If the text does not contain a procedural guide, respond with 'No procedural guide found.'

<{input}>





In [16]:
result = run_llm_chain(model_id=gpt_model,
              prompt = prompt,
              input = text)

In [17]:
print(result)

Reset Your Password

1. Navigate to the settings page.
2. Click the **Security** tab and select **Change Password**.
3. Enter your current password, then enter your new password twice to confirm it.
4. Click **Save Changes** to update your password.


# 🤔 **Strategies for Complex Queries**

The student-grading example in cell 22 uses all four strategies.

1. **Detail Steps & Format**: Outline each step to complete a task and specify the output format. Cell 22 lists the steps and the reply format.

2. **Break Down Tasks**: Like humans, the model can benefit from step-by-step instructions for complex tasks. Cell 22 asks the model to solve the problem itself before it checks the student's work.

3. **Logical Sequencing**: Request a logical flow or reasoning chain before the final answer. In cell 22, the "Actual solution" section comes before the "Student grade".

4. **Think Before Concluding**: Instruct the model to fully process the task before jumping to conclusions. Cell 22 tells the model not to grade the student until it has worked the problem.

The model reaches the correct answer in cell 26 (the student used 100x for maintenance instead of 10x). Still, check the working yourself: a confident reply is not proof that the model is right.

The reply in cell 26 repeats the question and the student's solution because the format in cell 22 asks for them. This is the model following the format exactly.

In [18]:
template = """Your task is to determine if a student's solution is correct or incorrect.\

To solve the problem do the following:

- First, work out your own solution to the problem including the final total.

- Then compare your solution to the student's solution and evaluate if the student's \
solution is correct or not.

Don't decide if the student's solution is correct until you have done the problem yourself.

Question: ```{input}```

"""

suffix = """Respond with the following format:

Question: ```question here```

Student's solution: ```student's solution here```

Actual solution: ```steps to work out the solution and your solution here```

Is the student's solution the same as actual solution just calculated:```yes or no```

Student grade: ```correct or incorrect```

Think step-by-step, and provide your solution to the question.

My solution is: 
"""


question = """```I'm building a solar power installation and I need help working out the financials:
- Land costs $100 / square foot
- I can buy solar panels for $250 / square foot
- I negotiated a contract for maintenance that will cost me a flat $100k per year,\
and an additional $10 / square foot

What is the total cost for the first year of operations as a function of the number of square feet.```

Student's solution:```Let x be the size of the installation in square feet.

Costs:
1. Land cost: 100x
2. Solar panel cost: 250x
3. Maintenance cost: 100,000 + 100x

Total cost: 100x + 250x + 100,000 + 100x = 450x + 100,000```
"""


In [19]:
prompt = PromptTemplate.from_examples(
    examples=[template],
    input_variables=["input"],
    prefix = "",
    suffix=suffix
    )

In [20]:
prompt.pretty_print()



Your task is to determine if a student's solution is correct or incorrect.
To solve the problem do the following:

- First, work out your own solution to the problem including the final total.

- Then compare your solution to the student's solution and evaluate if the student's solution is correct or not.

Don't decide if the student's solution is correct until you have done the problem yourself.

Question: ```{input}```



Respond with the following format:

Question: ```question here```

Student's solution: ```student's solution here```

Actual solution: ```steps to work out the solution and your solution here```

Is the student's solution the same as actual solution just calculated:```yes or no```

Student grade: ```correct or incorrect```

Think step-by-step, and provide your solution to the question.

My solution is: 



In [21]:
result = run_llm_chain(model_id = gpt_model,
              prompt = prompt,
              input = question)



In [22]:
print(result)

Question: ```I'm building a solar power installation and I need help working out the financials:
- Land costs $100 / square foot
- I can buy solar panels for $250 / square foot
- I negotiated a contract for maintenance that will cost me a flat $100k per year,and an additional $10 / square foot

What is the total cost for the first year of operations as a function of the number of square feet.```

Student's solution: ```Let x be the size of the installation in square feet.

Costs:
1. Land cost: 100x
2. Solar panel cost: 250x
3. Maintenance cost: 100,000 + 100x

Total cost: 100x + 250x + 100,000 + 100x = 450x + 100,000```

Actual solution: ```Let x be the installation size in square feet.

Land costs $100x, panels cost $250x, and first-year maintenance costs $100,000 + $10x.

Total first-year cost = 100x + 250x + 10x + 100,000 = 360x + 100,000 dollars.```

Is the student's solution the same as actual solution just calculated:```no```

Student grade: ```incorrect```


# 🎯 **Balance Context and Clarity**

- Provide essential information, but avoid overloading with details.

- Focus on key points in your prompt.

- In a chat, the model can ask for more context when a request is unclear. The code cell below shows this: the prompt tells the model to ask a question for each missing detail instead of guessing.

In [23]:
template = """If the request below is missing a product name, key features, or target audience, ask one question for each missing item instead of guessing.

Request: ```{input}```
"""

prompt = PromptTemplate.from_examples(
    examples=[template],
    input_variables=["input"],
    prefix = "",
    suffix=""
    )

request = "Write a product description for my app."

result = run_llm_chain(model_id = gpt_model,
              prompt = prompt,
              input = request)
print(result)

What’s the app’s name?

What are its key features?

Who is the target audience?


# Summary

1. 🎯 **Clarity is Essential**: Use clear delimiters to avoid ambiguity and ensure precise responses from the model.

2. 🔍 **Request Structured Output**: Ask for data in organized formats like JSON for ease of interpretation and use.

3. 🔍 **Process for Verifying Conditions**:
   - **Spot the Criteria**: Identify specific requirements in the text.
   - **Assess Compliance**: Check if the text meets these criteria.
   - **Customized Reply**: Respond based on the fulfillment of conditions.
   - **Transparent Feedback**: Clearly communicate the outcome and reasoning.

4. 🤔 **Strategies for Complex Queries**:
   - **Detail Steps & Format**: Define steps and output format for complex tasks.
   - **Break Down Tasks**: Use step-by-step instructions for better processing.
   - **Logical Sequencing**: Ask for a logical flow before the final answer.
   - **Think Before Concluding**: Encourage thorough processing before conclusions.

5. 🎯 **Balance Context and Clarity**: Provide essential information without overloading, allowing the model to request more context if needed.

6. **Describe the Output You Want**: Name the format, keys, or steps you need (see cells 10 to 14 and 16 to 20). You can also show the model examples in the prompt, which is called few-shot prompting. The code cell below shows this with two example input and output pairs.

In [24]:
template = """Write a one-line tagline in the same style as the examples.

Product: Solar-powered lamp
Tagline: Light that charges while you sleep.

Product: Cloud-based note app
Tagline: Your notes, wherever you go.

Product: {input}
Tagline:"""

prompt = PromptTemplate.from_examples(
    examples=[template],
    input_variables=["input"],
    prefix = "",
    suffix=""
    )

product = "Smart water bottle that reminds you to drink"

result = run_llm_chain(model_id = gpt_model,
              prompt = prompt,
              input = product)
print(result)

A sip of water, right on time.


# 🧪 **Try It Yourself**

1. Change the number of movies in the prompt in cell 10, or rename one JSON key. Then re-run cells 10 to 14. What changes in the output?

2. In cell 22, change the student's solution in the `question` variable so that it is correct. Then re-run cells 22 to 26. Does the grade change?

Keep a note of what you changed and what you observed. Model outputs can vary between runs, so try each version more than once if you can.